***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)
    * 下一节： [8.5 延伸阅读与参考文献](8_5_further_reading_and_references.ipynb)

***

导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [ ]:
pass

In [ ]:
HTML('../style/code_toggle.html')

## 8.4 第三代校准（3GC）：方向依赖自校准

正如[第 7 章 &#10142;](../7_Observing_Systems/7_0_introduction.ipynb)所说明的那样，现代望远镜视场越来越大，这使得主波束变化、指向误差等方向依赖效应开始变得不可忽略。因此，我们已经不能再仅依赖方向无关的自校准方法（见[$\S$ 8.3 &#10142;](../8_Calibration/8_3_2gc.ipynb)）。原则上，方向依赖校准有许多不同路径可走，[$\S$ 8.4.1 &#10549;](#cal:sec:p_versus_h)会对其做进一步分类。

本节先聚焦一种很有代表性的方法：*差分增益*（differential gains，见 [<cite data-cite='Smirnov2011'>Revisiting the radio interferometer measurement equation-II.  Calibration and direction-dependent effects</cite> &#10548;](http://arxiv.org/abs/1101.1765)）。它是理解“方向依赖校准与方向无关校准到底有何不同”的一个非常直观的框架。

在[$\S$ 7.2 &#10142;](../7_Observing_Systems/7_2_rime.ipynb)中，我们曾见过如下形式的全天 RIME：

<p class=conclusion>
  <font size=4> <b>全天 RIME</b></font>
  <br>
  <br>
  \begin{equation}
  \mathbf{V}_{pq} = \mathbf{G}_p\mathbf{X}_{pq}\mathbf{G}_q^H.
  \end{equation}
</p>

这里，$\mathbf{V}_{pq}$ 是干涉仪测得的 $2\times2$ 相关矩阵，$\mathbf{X}_{pq}$ 是对应的 $2\times2$ 相干矩阵，而 $\mathbf{G}_p$ 与 $\mathbf{G}_q$ 是 G-Jones 天线矩阵。校准时，我们求解的正是这些 $\mathbf{G}$ 项，并据此去改正 $\mathbf{V}_{pq}$。在这一写法中，$\mathbf{X}_{pq}$ 通常还可写成
$\mathbf{X}_{pq} = \sum_s \mathbf{X}_{spq}$，
其中 $s$ 是源编号。也就是说，全天 RIME 默认假设污染可见度的误差与源在天空中的位置无关。

但一旦进入大视场情形，这个假设就会失效。比如，仪器主波束在大视场内往往会显著随方向变化，并且通常还依赖时间和频率。对于像主波束这类方向依赖效应，我们有时可以尝试在 Jones 链中加入一个先验的 E-Jones 模型来处理（见 [<cite data-cite='Mitra2015'>Incorporation of antenna primary beam patterns in radio-interferometric data reduction to produce wide-field, high-dynamic-range images</cite> &#10548;](http://ieeexplore.ieee.org/Xplore/defdeny.jsp?url=http%3A%2F%2Fieeexplore.ieee.org%2Fstamp%2Fstamp.jsp%3Ftp%3D%26arnumber%3D7297163%26userType%3Dinst&denyReason=-134&arnumber=7297163&productsMatched=null&userType=inst)）。

但如果我们并不知道某个方向依赖效应背后的明确物理来源，那么还可以换一种思路：在方向无关增益之外，再给每个源额外引入一个差分增益项。数学上可写为：

<p class=conclusion>
  <font size=4> <b>加入差分增益</b></font>
  <br>
  <br>
  \begin{equation}
  \mathbf{V}_{pq} = \mathbf{G}_p \left (\sum_s\Delta\mathbf{E}_{sp}\mathbf{X}_{spq}\Delta\mathbf{E}_{sq}^H \right) \mathbf{G}_q^H,
\end{equation}
</p>

其中 $\Delta\mathbf{E}_{sp}$ 和 $\Delta\mathbf{E}_{sq}^H$ 分别表示源 $s$ 在天线 $p$ 和 $q$ 上对应的差分增益项。接下来，我们就可以利用[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb)中介绍的最小二乘思想，同时估计方向无关增益和差分增益，并用这些解去改正 $\mathbf{V}_{pq}$。

<div class=warn>
<b>注意：</b> 这里为了说明差分增益方法，我们重新使用了全极化 RIME，而在[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb)中使用的是非极化记号。
</div><br>

<div class=warn>
<b>注意：</b> 差分增益必须谨慎使用，否则很容易因为自由参数过多而导致过拟合。
</div>

另一个自然的问题是：我们如何判断某个源是否需要额外引入差分增益因子？图 [8.4.1 &#10549;](#cal:fig:dir_dep) 可以帮助建立直观理解。

<a id='cal:fig:dir_dep'></a><img src='figures/dir_dep.png' width=40%> <!--\label{cal:fig:dir_dep}-->

**图 8.4.1**：哪些源可能需要额外的差分增益因子？

从图 [8.4.1 &#10549;](#cal:fig:dir_dep) 可以得到两点直观认识：

1. 在实际图像中，若某个源周围出现明显成像伪影，那么它往往是需要额外差分增益项的候选目标。图中被紫色区域包围的黑色源，就是这类例子；而黄色源则没有明显伪影，因此通常不需要额外的差分增益。
2. 一个源离视场中心越远，受方向依赖效应影响的概率通常也越大。

下面给出一个真正使用差分增益方法得到的射电图像例子。图 [8.4.2 &#10549;](#cal:fig:diff_smirnov) 展示了 JVLA 对 3C147 场的成像结果。大图是最终的处理结果，也就是已经应用了差分增益方法之后的图像。左上角插图是大图中某个子区域的放大；左下角插图则展示了同一子区域在应用差分增益之前的样子。

这个例子再次说明：那些除了方向无关增益之外还需要差分增益的源，通常都会伴随明显的成像伪影。在该示例中，这些伪影主要来自 JVLA 主波束旋转所引入的方向依赖误差。

<a id='cal:fig:diff_smirnov'></a><img src='figures/dde_oleg.png' width=100%> <!--\label{cal:fig:diff_smirnov}-->

**图 8.4.2**：差分增益方法应用于 3C147 场后的结果（图片由 **O.M. Smirnov** 提供）。

### 8.4.1 基于物理模型的方法与启发式方法 <a id='cal:sec:p_versus_h'></a> <!--\label{cal:sec:p_versus_h}-->

总体来看，3GC 可以分为两大类：*基于物理模型的方法* 和 *启发式方法*。

如果我们知道某个方向依赖效应背后的物理来源，就可以采用基于物理模型的校准方法。其做法通常是先根据该物理过程构造一个参数化模型，然后在校准时估计这个模型中的参数，并利用求得的参数去改正观测可见度。有些方向依赖效应甚至在观测前就是已知的，这时校准的重点就变成了如何把这些先验信息正确地纳入数据处理中。典型例子包括：

<p class=conclusion>
  <font size=4> <b>3GC：基于物理模型的方法</b></font>
  <br>
  <br>
&bull; <em>Pointing-selfcal</em>： [<cite data-cite='Bhatnagar2004'>EVLA Memo 84. Solving for the antenna based pointing errors</cite> &#10548;](http://www.aoc.nrao.edu/evla/geninfo/memoseries/evlamemo84.ps) <br>
&bull; <em>Kalman filter</em>： [<cite data-cite='Tasse2014'>Nonlinear Kalman filters for calibration in radio interferometry</cite> &#10548;](http://arxiv.org/abs/1403.6308)<br> 
&bull; <em>Primary beam</em>： [<cite data-cite='Mitra2015'>Incorporation of antenna primary beam patterns in radio-interferometric data reduction to produce wide-field, high-dynamic-range images</cite> &#10548;](http://ieeexplore.ieee.org/Xplore/defdeny.jsp?url=http%3A%2F%2Fieeexplore.ieee.org%2Fstamp%2Fstamp.jsp%3Ftp%3D%26arnumber%3D7297163%26userType%3Dinst&denyReason=-134&arnumber=7297163&productsMatched=null&userType=inst)
</p>

即便方向依赖效应的来源是已知的，对其进行改正仍然不简单。下面列出一些常用来处理“已知方向依赖效应”的方法：

<p class=conclusion>
  <font size=4> <b>3GC：校正已知方向依赖效应的方法</b></font>
  <br>
  <br>
&bull; <em>Facetting</em>： [<cite data-cite='Cornwell1992'>Radio-interferometric imaging of very large fields-The problem of non-coplanar arrays</cite> &#10548;](http://adsabs.harvard.edu/abs/1992A%26A...261..353C)<br>
&bull; <em>AW-projection</em>： [<cite data-cite='Batnagar2008'>Correcting direction-dependent gains in the deconvolution of radio interferometric images</cite> &#10548;](http://arxiv.org/abs/0805.0834)
</p>

另一类则是启发式方法。在这种思路里，我们并不知道某个方向依赖误差背后的明确物理来源，因此不会先构造物理模型，而是引入若干自由参数，并按照某种用户定义的经验准则去优化它们。常见的 3GC 启发式方法包括：

<p class=conclusion>
  <font size=4> <b>3GC：启发式方法</b></font>
  <br>
  <br>
&bull; *Peeling*： [<cite data-cite='Noordam2004'>LOFAR calibration challenges</cite> &#10548;](http://proceedings.spiedigitallibrary.org/proceeding.aspx?articleid=847375)<br>
&bull; <em>Differential gains</em>： [<cite data-cite='Smirnov2011'>Revisiting the radio interferometer measurement equation-II.  Calibration and direction-dependent effects</cite> &#10548;](http://arxiv.org/abs/1101.1765)<br>
&bull; <em>Clustered calibration</em>： [<cite data-cite='Kazemi2013'>Clustered calibration: an improvement to radio interferometric direction-dependent self-calibration</cite> &#10548;](http://arxiv.org/abs/1301.0633)
</p>

在得到这些启发式解之后，我们有时还会尝试再反过来给它们拟合一个物理模型，从而把经验性校准结果重新解释成更具体的物理量。典型例子包括：

<p class=conclusion>
  <font size=4> <b>3GC：对启发式解再做物理拟合</b></font>
  <br>
  <br>
&bull; <em>SPAM（Source Peeling and Atmospheric Modelling）</em>： [<cite data-cite='Intema2009'>Ionospheric calibration of low frequency radio interferometric observations using the peeling scheme-I. Method description and first results</cite> &#10548;](http://arxiv.org/abs/0904.3975) <br> 
&bull; <em>Primary beam shapes</em>： [<cite data-cite='Yatawatta2013'>Estimation of radio interferometer beam shapes using Riemannian optimization</cite> &#10548;](http://arxiv.org/abs/1209.4236)
</p>

### 8.4.2 求解器的发展

正如[$\S$ 8.2 &#10142;](../8_Calibration/8_2_1gc.ipynb)所提到的那样，最小二乘求解器（见[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb)）真正流行起来，是在自校准时代到来之后。此后，人们又发展出了许多改进版和替代型求解器。下面列出一些较新的代表性工作：

<p class=conclusion>
  <font size=4> <b>求解器的发展</b></font>
  <br>
  <br>
&bull; <em>Eigendecomposition</em>： [<cite data-cite='Boonstra2003'>Gain calibration methods for radio telescope arrays</cite> &#10548;](http://ieeexplore.ieee.org/xpl/freeabs_all.jsp?arnumber=1145704&abstractAccess=no&userType=inst)<br>
&bull; <em>SAGEcal</em>： [<cite data-cite='Kazemi2011'>Radio interferometric calibration using the SAGE algorithm</cite> &#10548;](http://arxiv.org/abs/1012.1722)<br>
&bull; <em>Robust calibration</em>： [<cite data-cite='Kazemi2013robust'>Robust radio interferometric calibration using the t-distribution</cite> &#10548;](http://arxiv.org/abs/1307.5040)<br>
&bull; <em>StEFCal</em>： [<cite data-cite='Salvini2014'>Fast gain calibration in radio astronomy using alternating direction implicit methods: Analysis and applications</cite> &#10548;](http://arxiv.org/abs/1410.2101)<br>
&bull; <em>Riemann-Manifold</em>： [<cite data-cite='Yatawatta2013'>Radio interferometric calibration using a Riemannian manifold</cite> &#10548;](http://arxiv.org/abs/1303.1029)<br>
&bull; <em>Blind Calibration</em>： [<cite data-cite='Kazemi2015'>Blind calibration for radio interferometry using convex optimization</cite> &#10548;](http://ieeexplore.ieee.org/xpl/freeabs_all.jsp?arnumber=7330285&abstractAccess=no&userType=inst)<br>
&bull; <em>Complex Optimization</em>： [<cite data-cite='Smirnov2015'>Radio interferometric gain calibration as a complex optimization problem</cite> &#10548;](http://arxiv.org/abs/1502.06974)<br>
&bull; <em>Kalman filter</em>： [<cite data-cite='Tasse2014'>Nonlinear Kalman filters for calibration in radio interferometry</cite> &#10548;](http://arxiv.org/abs/1403.6308)
</p>

在上述方法中，StEFCal 往往被认为是近年来最重要的一个发展。我们会在[$\S$ 8.4.2.1 &#10142;](../8_Calibration/8_4_3gc.ipynb#cal:sec:stef)中更详细地介绍它。它之所以重要，部分原因就在于它把校准计算复杂度从 $O(N^3)$ 降低到了 $O(N^2)$。

#### 8.4.2.1 StEFCal <a id='cal:sec:stef'></a> <!--\label{cal:sec:stef}-->

StEFCal 是一种*交替方向隐式方法*（alternating direction implicit method）。它的基本做法是：先在固定 $\boldsymbol{\mathcal{G}}$ 的条件下求解 $\boldsymbol{\mathcal{G}}^H$，再在固定 $\boldsymbol{\mathcal{G}}^H$ 的条件下求解 $\boldsymbol{\mathcal{G}}$。从效果上看，这相当于把原本的校准问题局部线性化。<br><br>
<div class=warn>
<b>注意：</b> 这里又切回到了[$\S$ 8.1 &#10142;](../8_Calibration/8_1_calibration_least_squares_problem.ipynb)中所使用的非极化记号。
</div>

由于
$\boldsymbol{\mathcal{D}}- \boldsymbol{\mathcal{G}}\boldsymbol{\mathcal{M}}\boldsymbol{\mathcal{G}}^H$
是厄米的，因此这两个步骤实际上是等价的，于是只需要如下更新步骤：

$$\boldsymbol{\mathcal{G}}^{[i]} = \textrm{argmin}_{\boldsymbol{\mathcal{G}}}\left\|\boldsymbol{\mathcal{D}}- \boldsymbol{\mathcal{G}}^{[i-1]}\boldsymbol{\mathcal{M}}\boldsymbol{\mathcal{G}}^H\right\|.$$

于是进一步可写为：

$$\left\|\boldsymbol{\mathcal{D}} - \boldsymbol{\mathcal{Z}}^{[i]}\boldsymbol{\mathcal{G}}^H\right\| = \sqrt{\sum_{p}^N\boldsymbol{\mathcal{D}}_{:,p}-\boldsymbol{\mathcal{Z}}_{:,p}^{[i]}\left(g_p^{[i]}\right)^*},$$

其中
$\boldsymbol{\mathcal{Z}}^{[i]} = \boldsymbol{\mathcal{G}}^{[i]}\boldsymbol{\mathcal{M}}$。
这里用 $\boldsymbol{\mathcal{A}}_{:,p}$ 表示矩阵 $\boldsymbol{\mathcal{A}}$ 的第 $p$ 列。若再使用[正规方程法 &#10548;](http://mathworld.wolfram.com/NormalEquation.html)，就能直接得到：

<p class=conclusion>
  <font size=4> <b>StEFCal 更新公式</b></font>
  <br>
  <br>
  \begin{equation}
  g_p^{[i]} = \frac{\boldsymbol{\mathcal{D}}^H_{:,p}\boldsymbol{\mathcal{Z}}_{:,p}^{[i-1]}}{\left(\boldsymbol{\mathcal{Z}}_{:,p}^{[i-1]}\right)^H\boldsymbol{\mathcal{Z}}_{:,p}^{[i-1]}}.
  \end{equation}
</p>

利用这个更新公式，我们就可以迭代求得 $g_p$ 的最佳估计。算法通常在达到最大迭代次数，或者满足收敛条件时停止。<br><br>
<div class=warn>
<b>注意：</b> 在实际实现中，通常会把每一次偶数步迭代得到的增益解，替换成它与上一轮奇数步解的平均值，以提高稳定性。
</div>

***

下一节： [8.5 延伸阅读与参考文献](8_5_further_reading_and_references.ipynb)